# Padding Oracle — Notebook de ayudas (estudiantes)

Cuaderno guía para resolver el laboratorio paso a paso. **No contiene las respuestas**: las celdas marcadas con `TODO` las completas tú.

Requisito: oráculo arriba (local o Codespaces):
```bash
cd padding-oracle
docker compose up -d
```
En Codespaces usa el host por defecto (`127.0.0.1`). En la red docker original usa `ORACLE_HOST=10.9.0.80`.

## 📖 Lectura previa (10 min)

- Ejercicio gratuito de PentesterLab: https://pentesterlab.com/exercises/padding_oracle (<1h, nivel medio).
- Lee las secciones **Cipher Block Chaining**, **Padding**, **Padding Oracle** y **The theory**: su `I15 = 0x01 ^ E'7` es exactamente nuestro `D2[pos] = CC1[pos] ⊕ K`.
- Fíjate en su conclusión: el cifrado sin autenticación permite descifrar y **re-cifrar** datos a gusto (cifrado ≠ firma).
- Recomendado también su consejo de *manual exploitation*: escribir tu propia herramienta antes de automatizar.


In [ ]:
import os
import socket
from binascii import hexlify, unhexlify

HOST = os.getenv('ORACLE_HOST', '127.0.0.1')
PORT_L1 = int(os.getenv('ORACLE_PORT_L1', '5000'))
PORT_L2 = int(os.getenv('ORACLE_PORT_L2', '6000'))

def xor(a, b):
    return bytearray(x ^ y for x, y in zip(a, b))

class PaddingOracle:
    """Cliente TCP del oráculo: devuelve 'Valid' / 'Invalid'."""
    def __init__(self, host, port):
        self.s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.s.connect((host, port))
        self.ctext = unhexlify(self.s.recv(8192).decode().strip())
    def decrypt(self, ctext: bytes):
        self.s.send(hexlify(ctext) + b'\n')
        return self.s.recv(4096).decode().strip()
    def close(self):
        self.s.close()

In [ ]:
# Ayuda: mira lo que entrega el oráculo ANTES de atacar.
# Pregúntate: ¿cuántos bytes hay? ¿cuántos bloques de 16 B son (sin contar el IV)?
oracle = PaddingOracle(HOST, PORT_L1)
raw = bytearray(oracle.ctext)
IV, C1, C2 = raw[0:16], raw[16:32], raw[32:48]
print(f'IV: {IV.hex()}')
print(f'C1: {C1.hex()}')
print(f'C2: {C2.hex()}')
print(f'total: {len(raw)} bytes = IV + ? bloques')

## Ayudas Task 1 — padding PKCS#7

1. Crea archivos de 5, 10 y 16 bytes con `echo -n` (sin `-n` se cuela un `\n` extra).
2. Cifra con `openssl enc -aes-128-cbc -e` y descifra con `-d -nopad` para **ver** el relleno (sin `-nopad` OpenSSL lo quita solo).
3. Muestra el descifrado con `xxd` (los bytes de relleno no siempre son imprimibles).
4. Pregunta clave del informe: si el archivo ya mide 16 B, ¿por qué el cifrado ocupa 32 B? ¿Qué pasaría al quitar el padding de un mensaje que termina en `0x01` si no existiera ese bloque extra?

El script `task1.sh` ya hace los 4 pasos; ejecútalo y anota qué relleno ves en cada caso.

In [ ]:
!bash task1.sh

## Ayudas Nivel 1 — ideas antes de programar

- En CBC el descifrado es `P2 = C1 ⊕ D2`, donde `D2 = Dec(K, C2)` es desconocido. Si hallas `D2`, hallas `P2`.
- Puedes enviar `IV + CC1 + C2` con el `CC1` que quieras. El oráculo calcula `X = D2 ⊕ CC1` y solo te dice si `X` tiene padding válido.
- Padding válido de longitud `K` = los últimos `K` bytes de `X` valen `K` (ej. `K=1` → último byte `0x01`).
- Estrategia por byte (de atrás hacia adelante): para hallar el byte `pos = 16-K`, primero **fija** los bytes ya conocidos al valor que forme padding `K`: `CC1[j] = D2[j] ⊕ K`. Luego prueba los 256 valores de `CC1[pos]`; el que dé `Valid` revela `D2[pos] = CC1[pos] ⊕ K`.
- Con los últimos 6 bytes ya ves padding + cola del mensaje (mínimo del lab); con los 16 lo tienes completo.

In [ ]:
# Ejemplo dado (K=1): halla D2[15].
# El valor que dé 'Valid' cumple: D2[15] = valor ⊕ 0x01. Anótalo: lo necesitarás.
K = 1
CC1 = bytearray(16)
for i in range(256):
    CC1[16 - K] = i
    status = oracle.decrypt(bytes(IV) + bytes(CC1) + bytes(C2))
    if status == 'Valid':
        print(f'Válido: i = 0x{i:02x}')
        print(f'CC1: {CC1.hex()}')

## ✏️ Ejercicio 1 — generaliza a K=1..16 (mínimo 6 últimos bytes)

Ayudas:

1. Envuelve el ejemplo en `for K in range(1, 17)` con `pos = 16 - K`.
2. Al inicio de cada vuelta fija el sufijo ya hallado: `CC1[j] = D2[j] ^ K` para `j > pos`.
3. Guarda **todos** los candidatos `Valid` de cada `K` en una lista e imprímela (ver el nº de candidatos te ayuda a depurar).
4. Calcula `D2[pos]` y el byte de plano `P2[pos] = C1[pos] ^ D2[pos]`. ¿Es imprimible? (`chr` si está entre 32 y 127).
5. Caso trampa en `K=1`: a veces hay 2 candidatos (el padding `01` real y un falso `02 02`). Ayuda: altera un byte vecino del sufijo (p. ej. `CC1[14] ^= 0x01`) y re-pregunta; el `01` real sigue dando `Valid` y el falso cae a `Invalid`.
6. Lleva la cuenta de consultas: como máximo 256 por byte. ¿Cuántas usaste en total?

In [ ]:
# TODO: completa el ataque byte a byte y calcula P2 = C1 ⊕ D2.
D2 = bytearray(16)
CC1 = bytearray(16)
queries = 0

for K in range(1, 17):
    pos = 16 - K
    ...  # 1) fija el sufijo conocido a padding K
    ...  # 2) prueba los 256 valores de CC1[pos], cuenta consultas, guarda candidatos
    ...  # 3) elige el valor válido (ojo al caso trampa de K=1) y calcula D2[pos]

P2 = xor(C1, D2)
print('D2:', D2.hex())
print('P2:', P2.hex())
print('consultas:', queries)

## ✏️ Ejercicio 2 (opcional) — deriva P1

Ayuda: es el mismo ataque pero sobre el bloque `C1`. ¿Qué bloque hace de "previo" para `C1`? Pista: el payload a enviar ya no lleva el `IV` original delante; constrúyelo como `CC0 + C1` y al final calcula `P1 = IV_original ⊕ D1`.

## ✏️ Ejercicio 3 — Nivel 2 automatizado (puerto 6000)

Ayudas:

1. **Una sola conexión** durante todo el ataque: cada conexión nueva regenera clave e IV. Abre el socket una vez y reutilízalo.
2. Lee el IV + N bloques del primer mensaje y ataca de **atrás hacia adelante**.
3. Para el bloque `i`, el "previo" es `C[i-1]` (o el `IV` si `i == 0`). El payload es `IV_fijo + CC + C[i]`… salvo para el bloque 0, donde `CC` ocupa el lugar del IV: `CC + C[0]`.
4. Reutiliza la lógica del Ejercicio 1 en una función `ataca_bloque(previo, objetivo, ...) → (D, P)`.
5. Al unir los bloques quita el padding PKCS#7 (último byte `n` → elimina `n` bytes) y verifica que el mensaje tenga sentido.
6. Incluye en el informe el nº total de consultas y una captura de la salida.

In [ ]:
# TODO: automatiza el Nivel 2. Esqueleto (no lo ejecutes hasta completarlo).
# oracle2 = PaddingOracle(HOST, PORT_L2)  # ¡una sola conexión!
...  # 1) separa IV y bloques
...  # 2) define ataca_bloque() reutilizando el Ejercicio 1
...  # 3) recorre los bloques al revés, guarda cada P
...  # 4) une, quita padding, imprime mensaje y consultas
# oracle2.close()

## ✅ Verificación con PadBuster (solo para contrastar)

PadBuster (https://github.com/GDSSecurity/PadBuster, Perl) automatiza este mismo ataque sobre aplicaciones **HTTP**.
No se conecta a nuestro oráculo TCP directamente: úsalo en el ejercicio de PentesterLab para verificar que entiendes el comportamiento del oráculo.

```bash
git clone https://github.com/GDSSecurity/PadBuster && cd PadBuster
# 1) Descifrar la cookie (tamaño de bloque 8 en ese ejercicio, DES):
perl padbuster.pl http://<IP-del-lab> "<cookie>" 8 -cookies "auth=<cookie>"
# 2) Re-cifrar un mensaje a tu elección (p. ej. forjar admin):
perl padbuster.pl http://<IP-del-lab> "<cookie>" 8 -cookies "auth=<cookie>" -plaintext "user=admin"
```

Si PadBuster descifra la cookie y tu ataque manual contra el oráculo SEED avanza byte a byte con la misma lógica, vas bien.
⚠️ El informe se califica sobre **tu propio código** contra el oráculo SEED, no sobre salidas de PadBuster.

## Checklist de entrega

- [ ] Task 1: salidas `xxd` de 5/10/16 B + respuesta al bloque extra.
- [ ] Nivel 1: tabla por byte (`K`, `CC1` válido, `D2`, `P2`) con al menos 6 bytes + explicación del caso trampa.
- [ ] Nivel 2: tu script + salida con todos los bloques y nº de consultas.
- [ ] Conclusión: por qué el oráculo filtra información y mitigación (`Encrypt-then-MAC` / AES-GCM).

Cierra los sockets y apaga al terminar: `oracle.close()` y `docker compose down`.